In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────
# Ministral requires uninstall+reinstall to fix FP8 dequantize bug.
# LLaMA, Qwen, Gemma work with --upgrade only.
# We do the safest install that works for all 4 models.
!pip uninstall -q -y transformers
!pip install -q "git+https://github.com/huggingface/transformers"
!pip install -q accelerate pillow scikit-learn openpyxl numpy pandas mistral-common
import warnings, logging
print("Install complete.")
# !! RESTART RUNTIME after this cell, then run from Cell 2 !!

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 142.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 145.8 MB/s eta 0:00:00
Install complete.


In [ ]:
from huggingface_hub import login
login("hf_REDACTED_ROTATE_THIS_TOKEN")

In [ ]:
# ── Cell 2: Mount Drive & paths ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os

ANNOT_CSV   = "/content/drive/MyDrive/DIPA2/dataset/annotations.csv"
IMAGE_DIR   = "/content/drive/MyDrive/DIPA2/dataset/images"
CAPTION_DIR = "/content/drive/MyDrive/DIPA2/new_exps/caption_gen_dipa3/captions_v1"

# Each model gets its own output folder
OUTPUT_BASE = "/content/drive/MyDrive/DIPA2/multimodal_results"
os.makedirs(OUTPUT_BASE, exist_ok=True)

# HuggingFace token — required for LLaMA (gated model)
HF_TOKEN = "hf_REDACTED_ROTATE_THIS_TOKEN"

print(f"Annotations : {ANNOT_CSV}")
print(f"Images      : {IMAGE_DIR}")
print(f"Captions    : {CAPTION_DIR}")
print(f"Output base : {OUTPUT_BASE}")

Mounted at /content/drive
Annotations : /content/drive/MyDrive/DIPA2/dataset/annotations.csv
Images      : /content/drive/MyDrive/DIPA2/dataset/images
Captions    : /content/drive/MyDrive/DIPA2/new_exps/caption_gen_dipa3/captions_v1
Output base : /content/drive/MyDrive/DIPA2/multimodal_results


In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────
import os, json, re, time, random, gc
import warnings, transformers
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")
from pathlib import Path
from PIL import Image
from collections import Counter
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    :", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-40GB
VRAM    : 42.4 GB


In [ ]:
# ── Cell 4: Configuration ─────────────────────────────────────────
# All 4 models run sequentially in one session.
# Each model saves to its own subfolder under OUTPUT_BASE.
# Checkpoints are per-model-per-task — resume works correctly.

MODELS_TO_RUN = ["qwen", "gemma", "ministral", "llama"]
# Set to subset if you want to run only specific models, e.g. ["qwen", "gemma"]

NUM_RUNS         = 3
TEMPERATURES     = [0.1, 1.0]
MAX_IMAGE_PX     = 1024
MAX_TOKENS_T3    = 150
MAX_TOKENS_T4    = 30
RUN_SEEDS        = [0, 42, 84]
CHECKPOINT_EVERY = 50

DATASET_NAME = "DIPA2-Multimodal"
DATASET_SLUG = "dipa2_multimodal"

print(f"Models to run : {MODELS_TO_RUN}")
print(f"Temperatures  : {TEMPERATURES}")
print(f"Tasks         : T3 (attribute recognition) + T4 (binary source attribution)")
print(f"Note          : T1/T2 excluded — DIPA2 has no safe images (all private)")

Models to run : ['qwen', 'gemma', 'ministral', 'llama']
Temperatures  : [0.1, 1.0]
Tasks         : T3 (attribute recognition) + T4 (binary source attribution)
Note          : T1/T2 excluded — DIPA2 has no safe images (all private)


In [ ]:
# ── Cell 5: Privacy Taxonomy (hierarchical, paper Figure 2) ───────
PRIVACY_TAXONOMY = {
    "Biometric Data":                   {"examples": ["face","fingerprints","audio","iris","gait"]},
    "Children Images":                  {"examples": ["school events","playgrounds"]},
    "Financial Information":            {"examples": ["credit cards","checks","receipts"]},
    "HIPAA Data":                       {"examples": ["medical records","prescriptions","health devices","disabilities"]},
    "Legal Identifiers":                {"examples": ["names","IDs","passports","addresses"]},
    "Digital Identifiers":              {"examples": ["email","phone number","passwords","computer screen content"]},
    "Personal Metadata (Demographics)": {"examples": ["gender","race","age","beliefs","occupation"]},
    "GPS Data":                         {"examples": ["gps data","live location"]},
    "Vehicle Information":              {"examples": ["license plates","vehicle ownership"]},
    "Nudity":                           {"examples": ["nudity","explicit content","adult imagery"]},
    "Violent/Unlawful Actions":         {"examples": ["criminal acts","weapons","vandalism","cigarettes"]},
    "Personal Context":                 {"examples": ["pets","home interior","family gatherings","personal items"]},
    "Location Identifiers":             {"examples": ["location photos","landmarks"]},
    "Background Individuals":           {"examples": ["passerby","bystanders","not clearly visible individuals"]},
}
VALID_CATEGORIES = set(PRIVACY_TAXONOMY.keys())

# DIPA2: 8 active categories per paper Table 8
DIPA2_ACTIVE_CATEGORIES = {
    "Biometric Data","Digital Identifiers","Legal Identifiers",
    "Violent/Unlawful Actions","Vehicle Information","Personal Context",
    "Location Identifiers","Background Individuals",
}

def get_taxonomy_string():
    lines = ["Taxonomy:"]
    lines.append("1. Biometric Data: -face -fingerprints -audio -iris -gait")
    lines.append("2. Children Images: -school events -playgrounds")
    lines.append("3. PII (Personal Identifiable Information):")
    lines.append("  3a. Financial Information: -credit cards -checks -receipts")
    lines.append("  3b. HIPAA Data: -medical records -prescriptions -health devices -disabilities")
    lines.append("  3c. Legal Identifiers: -names -IDs -passports -addresses")
    lines.append("  3d. Digital Identifiers: -email -phone number -passwords -computer screen content")
    lines.append("  3e. Personal Metadata (Demographics): -gender -race -age -beliefs -occupation")
    lines.append("  3f. GPS Data: -gps data -live location")
    lines.append("  3g. Vehicle Information: -license plates -vehicle ownership")
    lines.append("4. Legal Sensitivity Information:")
    lines.append("  4a. Nudity: -nudity -explicit content -adult imagery")
    lines.append("  4b. Violent/Unlawful Actions: -criminal acts -weapons -vandalism -cigarettes")
    lines.append("5. Personal Life:")
    lines.append("  5a. Personal Context: -pets -home interior -family gatherings -personal items")
    lines.append("  5b. Location Identifiers: -location photos -landmarks")
    lines.append("6. Background Individuals: -passerby -bystanders -not clearly visible individuals")
    return "\n".join(lines)

HIER_TO_CAT = {
    "1":"Biometric Data","2":"Children Images",
    "3a":"Financial Information","3b":"HIPAA Data","3c":"Legal Identifiers",
    "3d":"Digital Identifiers","3e":"Personal Metadata (Demographics)",
    "3f":"GPS Data","3g":"Vehicle Information",
    "4a":"Nudity","4b":"Violent/Unlawful Actions",
    "5a":"Personal Context","5b":"Location Identifiers",
    "6":"Background Individuals",
}
FLAT_INDEX_TO_CAT = {str(i+1): cat for i, cat in enumerate(PRIVACY_TAXONOMY.keys())}
print(f"Taxonomy         : {len(PRIVACY_TAXONOMY)} categories")
print(f"DIPA2 active     : {len(DIPA2_ACTIVE_CATEGORIES)} categories")
print(f"Active: {sorted(DIPA2_ACTIVE_CATEGORIES)}")

Taxonomy         : 14 categories
DIPA2 active     : 8 categories
Active: ['Background Individuals', 'Biometric Data', 'Digital Identifiers', 'Legal Identifiers', 'Location Identifiers', 'Personal Context', 'Vehicle Information', 'Violent/Unlawful Actions']


In [ ]:
# ── Cell 6: DIPA2 → Taxonomy mapping (paper Table 8) ─────────────
# Two key formats:
#   annotations.csv DIPACategory column: space-separated, Title Case → lowercased at load time
#   caption JSON label field: underscore-separated, lowercase
# Both are covered here.

DIPA2_TO_TAXONOMY = {
    # Biometric Data
    "person":               "Biometric Data",
    "finger":               "Biometric Data",
    # Digital Identifiers
    "machine":              "Digital Identifiers",
    "screen":               "Digital Identifiers",
    "electronic devices":   "Digital Identifiers",   # CSV format
    "electronic_devices":   "Digital Identifiers",   # caption format
    "electronic":           "Digital Identifiers",
    # Legal Identifiers
    "identity":             "Legal Identifiers",
    "printed material":     "Legal Identifiers",     # CSV format
    "printed_material":     "Legal Identifiers",     # caption format
    "printed materials":    "Legal Identifiers",     # plural CSV
    "printed_materials":    "Legal Identifiers",     # plural caption
    # Vehicle Information
    "vehicle plate":        "Vehicle Information",   # CSV format
    "vehicle_plate":        "Vehicle Information",   # caption format
    "license plate":        "Vehicle Information",
    "license_plate":        "Vehicle Information",
    # Violent/Unlawful Actions
    "cigarettes":           "Violent/Unlawful Actions",
    "cigarette":            "Violent/Unlawful Actions",
    # Personal Context
    "clothing":             "Personal Context",
    "book":                 "Personal Context",
    "table":                "Personal Context",
    "toy":                  "Personal Context",
    "home interior":        "Personal Context",      # CSV format
    "home_interior":        "Personal Context",      # caption format
    "cosmetics":            "Personal Context",
    "musical instrument":   "Personal Context",      # CSV format
    "musical_instrument":   "Personal Context",      # caption format
    "accessory":            "Personal Context",
    "pet":                  "Personal Context",
    "food":                 "Personal Context",
    # Location Identifiers
    "place identifier":     "Location Identifiers",  # CSV format
    "place_identifier":     "Location Identifiers",  # caption format
    "scenery":              "Location Identifiers",
    # Background Individuals
    "photo":                "Background Individuals",
}

# Verify all 8 active categories are reachable
reachable = set(DIPA2_TO_TAXONOMY.values()) & DIPA2_ACTIVE_CATEGORIES
missing   = DIPA2_ACTIVE_CATEGORIES - reachable
if missing:
    print(f"WARNING — unreachable: {missing}")
else:
    print(f"Mapping entries  : {len(DIPA2_TO_TAXONOMY)}")
    print(f"All 8 active categories reachable ✓")
    print(f"Reachable: {sorted(reachable)}")

Mapping entries  : 34
All 8 active categories reachable ✓
Reachable: ['Background Individuals', 'Biometric Data', 'Digital Identifiers', 'Legal Identifiers', 'Location Identifiers', 'Personal Context', 'Vehicle Information', 'Violent/Unlawful Actions']


In [ ]:
# ── Cell 7: Prompts ───────────────────────────────────────────────
# DIPA2: all images are private → no Safe option in Task 3 prompt.
# Task 3: flat output {"categories": [...]}
# Task 4: binary source attribution {"image_private": "Yes/No", "caption_private": "Yes/No"}
# Both tasks inject caption. Two variants: explicit (PII caption) / no_leak (clean).

def get_prompt_task3(caption: str) -> str:
    tax = get_taxonomy_string()
    return (
        'You are given a social media post consisting of an image and an accompanying caption.\n'
        f'Caption: "{caption}"\n\n'
        'Task: Identify which privacy categories are violated by this post based on the taxonomy. '
        'Include all applicable categories found in both the image and the caption.\n'
        f'{tax}\n'
        'Instructions:\n'
        '- Do not justify your answer.\n'
        '- Output ONLY a JSON object. No explanation, no markdown.\n'
        '- Output format: {"categories": [Names of the categories]}\n'
        '- Use exact category names from the taxonomy above.\n\n'
        'Now classify the post and output only the JSON.'
    )

def get_prompt_task4(caption: str) -> str:
    tax = get_taxonomy_string()
    return (
        'You are given a social media post consisting of an image and an accompanying caption.\n'
        f'Caption: "{caption}"\n\n'
        'Task: Determine the source of any privacy violation in this post based on the taxonomy.\n'
        f'{tax}\n'
        'Instructions:\n'
        '- Does the IMAGE alone contain private information from the taxonomy? Answer Yes or No.\n'
        '- Does the CAPTION alone contain private information from the taxonomy? Answer Yes or No.\n'
        '- Do not justify your answer.\n'
        '- Output ONLY a JSON object. No explanation, no markdown.\n'
        '- Output format: {"image_private": "Yes" or "No", "caption_private": "Yes" or "No"}\n\n'
        'Now classify and output only the JSON.'
    )

print("Prompts defined.")

Prompts defined.


In [ ]:
# ── Cell 8: Dataset loader ────────────────────────────────────────
# Selection: 265 images with unanimous annotator agreement (4/4).
# All 265 are private — no safe class in DIPA2.
#
# GT design:
#   Task 3 explicit: gt_annotation | gt_cap_explicit
#   Task 3 no_leak : gt_annotation only
#   Task 4 image   : gt_image_private = True always (all DIPA2 images private)
#   Task 4 caption explicit: gt_cap_private = True if caption adds labels
#   Task 4 caption no_leak : gt_cap_private = False always

def load_dataset(annot_csv, image_dir, caption_dir):
    df = pd.read_csv(annot_csv)

    # Build image file index (stems for both hex and numeric IDs)
    image_index = {os.path.splitext(e.name)[0]: e.path
                   for e in os.scandir(image_dir) if not e.name.startswith(".")}
    caption_index = {os.path.splitext(e.name)[0]: e.path
                     for e in os.scandir(caption_dir) if e.name.endswith(".json")}

    # Step 1: unanimous images
    privacy_per_image = (
        df[df["DIPACategory"].str.strip().str.lower() != "others"]
        .groupby("imagePath").size()
    )
    unanimous_ids = set(privacy_per_image[privacy_per_image >= 4].index)
    print(f"Unanimous images : {len(unanimous_ids)}")

    # Step 2: taxonomy label lookup from annotations.csv
    sub = df[
        df["imagePath"].isin(unanimous_ids) &
        (df["DIPACategory"].str.strip().str.lower() != "others")
    ].copy()
    sub["taxonomy"] = sub["DIPACategory"].str.strip().str.lower().map(DIPA2_TO_TAXONOMY)
    sub = sub[sub["taxonomy"].notna()]
    tax_lookup = sub.groupby("imagePath")["taxonomy"].apply(set).to_dict()

    samples = []; missing_img = 0; missing_cap = 0

    for img_path_key in unanimous_ids:
        img_id   = Path(img_path_key).stem
        img_path = image_index.get(img_id)
        if img_path is None: missing_img += 1; continue

        if img_id not in caption_index: missing_cap += 1; continue

        gt_annotation = tax_lookup.get(img_path_key, set()) & DIPA2_ACTIVE_CATEGORIES

        with open(caption_index[img_id]) as f: cap = json.load(f)
        caps         = cap.get("captions", {})
        cap_explicit = caps.get("explicit", "")
        cap_no_leak  = caps.get("no_leak",  "")

        # Caption-induced GT: only labels with source == "caption_induced"
        gt_cap_explicit = set()
        for lbl_entry in cap.get("labels", []):
            if lbl_entry.get("source") == "caption_induced":
                mapped = DIPA2_TO_TAXONOMY.get(lbl_entry.get("label","").lower())
                if mapped and mapped in DIPA2_ACTIVE_CATEGORIES:
                    gt_cap_explicit.add(mapped)

        samples.append({
            "id":                    img_id,
            "image_path":            img_path,
            "caption_explicit":      cap_explicit,
            "caption_no_leak":       cap_no_leak,
            "gt_annotation":         gt_annotation,
            "gt_cap_explicit":       gt_cap_explicit,
            "gt_combined_explicit":  gt_annotation | gt_cap_explicit,
            "gt_combined_noleak":    gt_annotation,
            "gt_image_private":      True,   # all DIPA2 images are private
            "gt_cap_private_explicit": len(gt_cap_explicit) > 0,
            "gt_cap_private_noleak":   False,
        })

    print(f"Loaded  : {len(samples)} samples")
    print(f"  Missing images   : {missing_img}")
    print(f"  Missing captions : {missing_cap}")
    print(f"  All private (no safe class in DIPA2)")

    from collections import Counter
    cat_counts = Counter()
    for s in samples:
        for c in s["gt_combined_explicit"]: cat_counts[c] += 1
    print(f"\nGT distribution (explicit combined):")
    for cat,cnt in sorted(cat_counts.items(), key=lambda x:-x[1]):
        print(f"  {cat:<40} {cnt}")
    return samples

dataset = load_dataset(ANNOT_CSV, IMAGE_DIR, CAPTION_DIR)
print(f"\nDataset ready: {len(dataset)} samples")

Unanimous images : 265
Loaded  : 265 samples
  Missing images   : 0
  Missing captions : 0
  All private (no safe class in DIPA2)

GT distribution (explicit combined):
  Biometric Data                           204
  Personal Context                         127
  Location Identifiers                     36
  Digital Identifiers                      35
  Vehicle Information                      21
  Legal Identifiers                        10
  Violent/Unlawful Actions                 4
  Background Individuals                   2

Dataset ready: 265 samples


In [ ]:
# ── Cell 9: Parsers, evaluation, checkpointing & runners ──────────
# All model-agnostic logic — defined once, shared across all 4 models.

def _resolve_cat(c_str, valid_cats=None):
    if valid_cats is None: valid_cats = DIPA2_ACTIVE_CATEGORIES
    c_str = str(c_str).strip()
    if c_str.lower() == "safe": return None
    if c_str in valid_cats: return c_str
    m = re.match(r'^(\d+[a-z]?)\.?\s*(.*)$', c_str)
    if m:
        resolved = HIER_TO_CAT.get(m.group(1).lower())
        if resolved and resolved in valid_cats: return resolved
        name = m.group(2).strip()
        if name in valid_cats: return name
        for cat in valid_cats:
            if cat.lower() == name.lower(): return cat
    if re.match(r'^\d+$', c_str):
        resolved = FLAT_INDEX_TO_CAT.get(c_str)
        if resolved and resolved in valid_cats: return resolved
    for cat in valid_cats:
        if cat.lower() == c_str.lower(): return cat
    return None

def parse_task3(response):
    predicted = set()
    try:
        m = re.search(r'\{.*\}', response, re.DOTALL)
        if m:
            parsed = json.loads(m.group())
            cats = parsed.get("categories", [])
            if isinstance(cats, list):
                for c in cats:
                    if str(c).lower() == "safe": return set()
                    r = _resolve_cat(c)
                    if r: predicted.add(r)
            if predicted: return predicted
    except Exception: pass
    r = response.strip()
    if re.search(r'\bsafe\b', r, re.IGNORECASE): return set()
    for cat in DIPA2_ACTIVE_CATEGORIES:
        if re.search(r'\b' + re.escape(cat) + r'\b', r, re.IGNORECASE):
            predicted.add(cat)
    return predicted

def parse_task4(response):
    try:
        m = re.search(r'\{.*\}', response, re.DOTALL)
        if m:
            parsed = json.loads(m.group())
            img_val = str(parsed.get("image_private",   "no")).strip().lower()
            cap_val = str(parsed.get("caption_private", "no")).strip().lower()
            return {"image_private": img_val=="yes", "caption_private": cap_val=="yes"}
    except Exception: pass
    r = response.lower(); lines = r.split("\n")
    img_p = False; cap_p = False
    for line in lines:
        if "image"   in line: img_p = "yes" in line
        if "caption" in line: cap_p = "yes" in line
    return {"image_private": img_p, "caption_private": cap_p}

def majority_labels(all_runs, num_runs):
    counts = Counter(lbl for run in all_runs for lbl in run)
    return {cat for cat, cnt in counts.items() if cnt > num_runs/2}

def majority_task4(all_runs, num_runs):
    img_votes = [r["image_private"]   for r in all_runs]
    cap_votes = [r["caption_private"] for r in all_runs]
    return {"image_private": img_votes.count(True)>num_runs/2,
            "caption_private": cap_votes.count(True)>num_runs/2}

def evaluate_recognition(results, gt_key, pred_key):
    cat_metrics = {}
    for cat in DIPA2_ACTIVE_CATEGORIES:
        yt=[1 if cat in r[gt_key]   else 0 for r in results]
        yp=[1 if cat in r[pred_key] else 0 for r in results]
        sup=int(sum(yt))
        if sup==0: continue
        p,r,f1,_=precision_recall_fscore_support(yt,yp,average="binary",pos_label=1,zero_division=0)
        cat_metrics[cat]={"precision":round(p*100,2),"recall":round(r*100,2),
                          "f1":round(f1*100,2),"support":sup}
    f1v=[m["f1"] for m in cat_metrics.values()]
    pv =[m["precision"] for m in cat_metrics.values()]
    rv =[m["recall"]    for m in cat_metrics.values()]
    return {"category_metrics":cat_metrics,
            "macro_f1":round(np.mean(f1v),2) if f1v else 0.0,
            "macro_precision":round(np.mean(pv),2) if pv else 0.0,
            "macro_recall":round(np.mean(rv),2) if rv else 0.0}

def evaluate_task4_source(results, variant):
    # gt_cap_private stored without variant suffix in runner
    out = {}
    for channel,gt_key,pred_key in [
        ("image",   "gt_image_private",  "pred_image_private"),
        ("caption", "gt_cap_private",    "pred_caption_private"),
    ]:
        yt=[1 if r[gt_key]   else 0 for r in results]
        yp=[1 if r[pred_key] else 0 for r in results]
        sup=int(sum(yt))
        if sup==0:
            out[channel]={"precision":0,"recall":0,"f1":0,"support":0,
                          "note":"no positive GT"}; continue
        p,r,f1,_=precision_recall_fscore_support(yt,yp,average="binary",pos_label=1,zero_division=0)
        out[channel]={"precision":round(p*100,2),"recall":round(r*100,2),"f1":round(f1*100,2),
                      "support":sup,"pred_positive":int(sum(yp))}
    return out

def _ser(r):
    rc=dict(r)
    for k,v in rc.items():
        if isinstance(v,set): rc[k]=sorted(list(v))
    return rc

def _deser(r):
    rc=dict(r)
    for k in ("gt_annotation","gt_cap_explicit","gt_combined_explicit",
              "gt_combined_noleak","pred_labels"):
        if k in rc and isinstance(rc[k],list): rc[k]=set(rc[k])
    return rc

def make_ckpt_path(ckpt_dir, slug, temp, task, variant):
    temp_str=str(temp).replace(".","_")
    return os.path.join(ckpt_dir, f"{slug}_{temp_str}_{task}_{variant}.ckpt.json")

def save_checkpoint(results, ckpt_dir, slug, temp, task, variant):
    path=make_ckpt_path(ckpt_dir,slug,temp,task,variant)
    tmp=path+".tmp"
    with open(tmp,"w") as f:
        json.dump({"n":len(results),"results":[_ser(r) for r in results]},f,indent=2)
    os.replace(tmp,path)

def load_checkpoint(ckpt_dir, slug, temp, task, variant):
    path=make_ckpt_path(ckpt_dir,slug,temp,task,variant)
    if not os.path.exists(path): return []
    with open(path) as f: data=json.load(f)
    results=[_deser(r) for r in data["results"]]
    if results: print(f"    Resumed {task}/{variant}: {len(results)} done")
    return results

def run_task3(samples, temp, variant, ckpt_dir, slug, call_model):
    results=load_checkpoint(ckpt_dir,slug,temp,"task3",variant)
    done_ids={r["id"] for r in results}
    remaining=[s for s in samples if s["id"] not in done_ids]
    t_start=time.time()
    for idx,s in enumerate(remaining):
        caption=s["caption_explicit"] if variant=="explicit" else s["caption_no_leak"]
        prompt=get_prompt_task3(caption)
        gt_combined=s["gt_combined_explicit"] if variant=="explicit" else s["gt_combined_noleak"]
        run_preds=[]; raw_outputs=[]
        for run in range(NUM_RUNS):
            raw=call_model(s["image_path"],prompt,temp,MAX_TOKENS_T3,RUN_SEEDS[run])
            run_preds.append(parse_task3(raw)); raw_outputs.append(raw)
        final=majority_labels(run_preds,NUM_RUNS) & DIPA2_ACTIVE_CATEGORIES
        results.append({"id":s["id"],"gt_combined":gt_combined,"pred_labels":final,
                        "all_runs":[sorted(list(r)) for r in run_preds],"raw_outputs":raw_outputs})
        if len(results)%CHECKPOINT_EVERY==0: save_checkpoint(results,ckpt_dir,slug,temp,"task3",variant)
        if (idx+1)%10==0 or idx==0:
            el=time.time()-t_start
            print(f"    [{len(results):4d}/{len(samples)}]  {el:.0f}s  avg {el/(idx+1):.1f}s/img")
    save_checkpoint(results,ckpt_dir,slug,temp,"task3",variant)
    return results,time.time()-t_start

def run_task4(samples, temp, variant, ckpt_dir, slug, call_model):
    results=load_checkpoint(ckpt_dir,slug,temp,"task4",variant)
    done_ids={r["id"] for r in results}
    remaining=[s for s in samples if s["id"] not in done_ids]
    t_start=time.time()
    for idx,s in enumerate(remaining):
        caption=s["caption_explicit"] if variant=="explicit" else s["caption_no_leak"]
        prompt=get_prompt_task4(caption)
        gt_cap_priv=s["gt_cap_private_explicit"] if variant=="explicit" else s["gt_cap_private_noleak"]
        run_splits=[]; raw_outputs=[]
        for run in range(NUM_RUNS):
            raw=call_model(s["image_path"],prompt,temp,MAX_TOKENS_T4,RUN_SEEDS[run])
            split=parse_task4(raw)
            run_splits.append(split); raw_outputs.append(raw)
        final=majority_task4(run_splits,NUM_RUNS)
        results.append({
            "id":                  s["id"],
            "gt_image_private":    s["gt_image_private"],
            "gt_cap_private":      gt_cap_priv,
            "pred_image_private":  final["image_private"],
            "pred_caption_private":final["caption_private"],
            "all_runs":run_splits,"raw_outputs":raw_outputs,
        })
        if len(results)%CHECKPOINT_EVERY==0: save_checkpoint(results,ckpt_dir,slug,temp,"task4",variant)
        if (idx+1)%10==0 or idx==0:
            el=time.time()-t_start
            print(f"    [{len(results):4d}/{len(samples)}]  {el:.0f}s  avg {el/(idx+1):.1f}s/img")
    save_checkpoint(results,ckpt_dir,slug,temp,"task4",variant)
    return results,time.time()-t_start

def serialize_result(r):
    r2=dict(r)
    for k,v in r2.items():
        if isinstance(v,set): r2[k]=sorted(list(v))
    return r2

def save_results_json(all_results, output_dir, slug, dataset_slug, n_samples):
    ts=time.strftime("%Y%m%d_%H%M%S")
    path=os.path.join(output_dir,f"{slug}_{dataset_slug}_{ts}_all_tasks.json")
    combined={}
    for task_key,task_data in all_results.items():
        if not task_data: continue
        combined[task_key]={}
        for tk_,tvars in task_data.items():
            combined[task_key][tk_]={}
            for variant,vdata in tvars.items():
                rc=[serialize_result(r) for r in vdata["results"]]
                combined[task_key][tk_][variant]={"metrics":vdata["metrics"],
                    "elapsed":vdata["elapsed"],"n_samples":len(rc),"results":rc}
    combined["_meta"]={"slug":slug,"dataset":DATASET_NAME,"n_samples":n_samples,
                       "temperatures":TEMPERATURES,"num_runs":NUM_RUNS,"seeds":RUN_SEEDS,
                       "timestamp":ts,"tasks":"T3+T4 (T1/T2 excluded — all DIPA2 images private)",
                       "gt_design":{
                           "task3_explicit":"gt_annotation | gt_cap_explicit",
                           "task3_noleak":  "gt_annotation only",
                           "task4_gt_image_private":"True always (all DIPA2 images private)",
                           "task4_gt_cap_private_explicit":"True if caption adds labels",
                           "task4_gt_cap_private_noleak":"Always False",
                       }}
    with open(path,"w") as f: json.dump(combined,f,indent=2)
    print(f"  JSON → {path}")
    return path

def save_results_excel(all_results, output_dir, slug, dataset_slug):
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter
    ts=time.strftime("%Y%m%d_%H%M%S")
    wb=Workbook(); wb.remove(wb.active)
    HDR=PatternFill("solid",fgColor="1F3864"); BEST=PatternFill("solid",fgColor="E2EFDA")
    SUB=PatternFill("solid",fgColor="D6E4F0")
    HF=Font(color="FFFFFF",bold=True,size=11)
    THIN=Side(style="thin"); BDR=Border(left=THIN,right=THIN,top=THIN,bottom=THIN)
    C=Alignment(horizontal="center",vertical="center",wrap_text=True)
    L=Alignment(horizontal="left",vertical="center",wrap_text=True)
    def hdr(ws,row,col,val,w=None):
        ce=ws.cell(row=row,column=col,value=val); ce.fill=HDR; ce.font=HF
        ce.border=BDR; ce.alignment=C
        if w: ws.column_dimensions[get_column_letter(col)].width=w
    def cel(ws,row,col,val,bold=False,fill=None,align=C):
        ce=ws.cell(row=row,column=col,value=val); ce.font=Font(bold=bold)
        ce.border=BDR; ce.alignment=align
        if fill: ce.fill=fill

    # Recognition T3
    ws2=wb.create_sheet("Recognition T3")
    for col,(label,w) in enumerate([("Variant",12),("Temp",7),("Category",32),
        ("Precision %",12),("Recall %",12),("F1 %",11),("Support",9)],1):
        hdr(ws2,1,col,label,w)
    row=2
    for tk_,tvars in all_results.get("task3",{}).items():
        tv=tk_.replace("temp=","")
        for variant,vdata in tvars.items():
            m=vdata["metrics"]
            for cat,cm in sorted(m["category_metrics"].items(),key=lambda x:x[1]["f1"],reverse=True):
                cel(ws2,row,1,variant,align=L); cel(ws2,row,2,tv)
                cel(ws2,row,3,cat,align=L)
                cel(ws2,row,4,cm["precision"]); cel(ws2,row,5,cm["recall"])
                cel(ws2,row,6,cm["f1"],bold=True); cel(ws2,row,7,cm["support"])
                row+=1
            cel(ws2,row,3,"MACRO",bold=True,fill=BEST,align=L)
            cel(ws2,row,4,m["macro_precision"],bold=True,fill=BEST)
            cel(ws2,row,5,m["macro_recall"],bold=True,fill=BEST)
            cel(ws2,row,6,m["macro_f1"],bold=True,fill=BEST); row+=2

    # Source Attribution T4
    ws3=wb.create_sheet("Source Attribution T4")
    for col,(label,w) in enumerate([("Variant",12),("Temp",7),("Channel",12),
        ("Precision %",12),("Recall %",12),("F1 %",11),("Support",9),("Pred Positive",13)],1):
        hdr(ws3,1,col,label,w)
    row=2
    for tk_,tvars in all_results.get("task4",{}).items():
        tv=tk_.replace("temp=","")
        for variant,vdata in tvars.items():
            m=vdata["metrics"]
            for channel in ["image","caption"]:
                cm=m.get(channel,{})
                fill=SUB if channel=="caption" else None
                cel(ws3,row,1,variant,fill=fill); cel(ws3,row,2,tv,fill=fill)
                cel(ws3,row,3,channel,align=L,fill=fill)
                cel(ws3,row,4,cm.get("precision","-"),fill=fill)
                cel(ws3,row,5,cm.get("recall","-"),fill=fill)
                cel(ws3,row,6,cm.get("f1","-"),bold=True,fill=fill)
                cel(ws3,row,7,cm.get("support","-"),fill=fill)
                cel(ws3,row,8,cm.get("pred_positive","-"),fill=fill)
                row+=1
            row+=1

    excel_path=os.path.join(output_dir,f"{slug}_{dataset_slug}_{ts}_all_tasks.xlsx")
    wb.save(excel_path)
    print(f"  Excel → {excel_path}")

print("All shared functions defined ✓")

All shared functions defined ✓


In [ ]:
# ── Cell 10: Per-model runner ─────────────────────────────────────
# Loads model, runs T3+T4, saves JSON+Excel, unloads model.
# call_model is defined inside run_model() so each model's version
# is correctly scoped and garbage collected before the next model loads.

import importlib

def run_model(model_key):
    global model, processor

    MODEL_CONFIGS = {
        "qwen": {
            "model_id":   "Qwen/Qwen3-VL-8B-Instruct",
            "slug":       "qwen_qwen3-vl-8b",
            "output_dir": os.path.join(OUTPUT_BASE, "qwen"),
        },
        "gemma": {
            "model_id":   "google/gemma-3-4b-it",
            "slug":       "google_gemma-3-4b",
            "output_dir": os.path.join(OUTPUT_BASE, "gemma"),
        },
        "ministral": {
            "model_id":   "mistralai/Ministral-3-3B-Instruct-2512-BF16",
            "slug":       "mistralai_ministral-3-3b",
            "output_dir": os.path.join(OUTPUT_BASE, "ministral"),
        },
        "llama": {
            "model_id":   "meta-llama/Llama-3.2-11B-Vision-Instruct",
            "slug":       "meta_llama3-2-11b",
            "output_dir": os.path.join(OUTPUT_BASE, "llama"),
        },
    }

    cfg = MODEL_CONFIGS[model_key]
    MODEL_ID   = cfg["model_id"]
    slug       = cfg["slug"]
    output_dir = cfg["output_dir"]
    ckpt_dir   = os.path.join(output_dir, "checkpoints")
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(ckpt_dir,   exist_ok=True)

    print(f"\n{'='*65}")
    print(f"  MODEL: {MODEL_ID}")
    print(f"  Output: {output_dir}")
    print(f"{'='*65}")

    # ── Load model ────────────────────────────────────────────────
    def set_seed(seed):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    def prepare_image(image_path, max_px=MAX_IMAGE_PX):
        img = Image.open(image_path).convert("RGB")
        if max(img.size) > max_px:
            img.thumbnail((max_px, max_px), Image.Resampling.LANCZOS)
        return img

    if model_key == "qwen":
        from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
        processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
        model.eval()
        def call_model(image_path, prompt, temperature, max_new_tokens, seed=0):
            set_seed(seed)
            img = prepare_image(image_path)
            messages=[{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":prompt}]}]
            text=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
            inputs=processor(text=[text],images=[img],return_tensors="pt",padding=True).to(model.device)
            do_sample=temperature>0.05
            with torch.no_grad():
                out=model.generate(**inputs,max_new_tokens=max_new_tokens,
                    temperature=temperature if do_sample else None,do_sample=do_sample,
                    pad_token_id=processor.tokenizer.eos_token_id)
            gen=out[:,inputs["input_ids"].shape[1]:]
            resp=processor.batch_decode(gen,skip_special_tokens=True)[0].strip()
            del inputs,out,gen; torch.cuda.empty_cache(); return resp

    elif model_key == "gemma":
        from transformers import AutoModelForImageTextToText, AutoProcessor
        processor = AutoProcessor.from_pretrained(MODEL_ID)
        model = AutoModelForImageTextToText.from_pretrained(
            MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
        model.eval()
        def call_model(image_path, prompt, temperature, max_new_tokens, seed=0):
            set_seed(seed)
            img = prepare_image(image_path)
            messages=[{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":prompt}]}]
            text=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
            inputs=processor(text=text,images=[img],return_tensors="pt",padding=True).to(model.device)
            do_sample=temperature>0.05
            with torch.no_grad():
                out=model.generate(**inputs,max_new_tokens=max_new_tokens,
                    temperature=temperature if do_sample else None,do_sample=do_sample,
                    pad_token_id=processor.tokenizer.eos_token_id)
            gen=out[:,inputs["input_ids"].shape[1]:]
            resp=processor.batch_decode(gen,skip_special_tokens=True)[0].strip()
            del inputs,out,gen; torch.cuda.empty_cache(); return resp

    elif model_key == "ministral":
        from transformers import Mistral3ForConditionalGeneration, AutoProcessor
        processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
        model = Mistral3ForConditionalGeneration.from_pretrained(
            MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
        model.eval()
        def call_model(image_path, prompt, temperature, max_new_tokens, seed=0):
            set_seed(seed)
            img = prepare_image(image_path)
            messages=[{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":prompt}]}]
            text=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
            inputs=processor(text=[text],images=[img],return_tensors="pt",padding=True).to(model.device)
            do_sample=temperature>0.05
            with torch.no_grad():
                out=model.generate(**inputs,max_new_tokens=max_new_tokens,
                    temperature=temperature if do_sample else None,do_sample=do_sample,
                    pad_token_id=processor.tokenizer.eos_token_id)
            gen=out[:,inputs["input_ids"].shape[1]:]
            resp=processor.batch_decode(gen,skip_special_tokens=True)[0].strip()
            del inputs,out,gen; torch.cuda.empty_cache(); return resp

    elif model_key == "llama":
        from transformers import MllamaForConditionalGeneration, AutoProcessor
        processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
        model = MllamaForConditionalGeneration.from_pretrained(
            MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto",
            token=HF_TOKEN, trust_remote_code=True)
        model.eval()
        # CRITICAL LLaMA: {"type":"image"} no image key, text= not list, no padding
        def call_model(image_path, prompt, temperature, max_new_tokens, seed=0):
            set_seed(seed)
            img = prepare_image(image_path)
            messages=[{"role":"user","content":[{"type":"image"},{"type":"text","text":prompt}]}]
            text=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
            inputs=processor(text=text,images=[img],return_tensors="pt").to(model.device)
            do_sample=temperature>0.05
            with torch.no_grad():
                out=model.generate(**inputs,max_new_tokens=max_new_tokens,
                    temperature=temperature if do_sample else None,do_sample=do_sample,
                    pad_token_id=processor.tokenizer.eos_token_id)
            gen=out[:,inputs["input_ids"].shape[1]:]
            resp=processor.batch_decode(gen,skip_special_tokens=True)[0].strip()
            del inputs,out,gen; torch.cuda.empty_cache(); return resp

    if torch.cuda.is_available():
        used=torch.cuda.memory_allocated()/1e9
        total=torch.cuda.get_device_properties(0).total_memory/1e9
        print(f"  VRAM: {used:.1f} GB / {total:.1f} GB")
    print(f"  Model ready.")

    # ── Run T3 + T4 ───────────────────────────────────────────────
    all_results = {"task3":{}, "task4":{}}
    model_start = time.time()

    for temp in TEMPERATURES:
        tk_ = f"temp={temp}"
        print(f"\n  ─ TEMPERATURE: {temp} ─")
        all_results["task3"][tk_] = {}
        all_results["task4"][tk_] = {}

        for variant in ["explicit","no_leak"]:
            print(f"\n  ▶ Task 3 [{variant}]")
            r3,e3=run_task3(dataset,temp,variant,ckpt_dir,slug,call_model)
            m3=evaluate_recognition(r3,"gt_combined","pred_labels")
            all_results["task3"][tk_][variant]={"results":r3,"metrics":m3,"elapsed":e3}
            print(f"    Macro F1: {m3['macro_f1']}%  P: {m3['macro_precision']}%  R: {m3['macro_recall']}%")
            for cat,cm in sorted(m3["category_metrics"].items(),key=lambda x:x[1]["f1"],reverse=True):
                print(f"      {cat:<40} F1={cm['f1']:5.1f}%  n={cm['support']}")

            print(f"\n  ▶ Task 4 [{variant}]")
            r4,e4=run_task4(dataset,temp,variant,ckpt_dir,slug,call_model)
            m4=evaluate_task4_source(r4,variant)
            all_results["task4"][tk_][variant]={"results":r4,"metrics":m4,"elapsed":e4}
            for ch,cm in m4.items():
                print(f"    {ch.upper()} — F1: {cm['f1']}%  P: {cm['precision']}%  R: {cm['recall']}%  support={cm['support']}")

    # ── Save ──────────────────────────────────────────────────────
    print(f"\n  Saving results...")
    save_results_json(all_results, output_dir, slug, DATASET_SLUG, len(dataset))
    save_results_excel(all_results, output_dir, slug, DATASET_SLUG)
    elapsed=time.time()-model_start
    print(f"  Total: {elapsed:.0f}s ({elapsed/60:.1f} min)")

    # ── Unload model to free VRAM ─────────────────────────────────
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  Model unloaded. VRAM freed.")

print("run_model() defined ✓")

run_model() defined ✓


In [ ]:
# ── Cell 11: MAIN — Run all models sequentially ───────────────────
# Each model loads, runs T3+T4 across both temps and variants,
# saves JSON+Excel to its own subfolder, then unloads.
# Total DIPA2 = 265 images × 2 tasks × 2 variants × 2 temps × 3 runs
# ≈ 6360 inference calls. At ~3-5s/image: 5-9 hours total for all 4 models.
# Checkpoints save every 50 images — safe to resume per model.

print("\n" + "="*65)
print(f"DIPA2 Multimodal — All 4 Models")
print(f"Dataset: {len(dataset)} samples | Tasks: T3, T4 | Variants: explicit, no_leak")
print(f"Models to run: {MODELS_TO_RUN}")
print("="*65)

pipeline_start = time.time()

for model_key in MODELS_TO_RUN:
    try:
        run_model(model_key)
    except Exception as e:
        print(f"\nERROR on {model_key}: {e}")
        import traceback; traceback.print_exc()
        print(f"Continuing to next model...")
        try:
            del model, processor
            gc.collect(); torch.cuda.empty_cache()
        except: pass

total_elapsed = time.time() - pipeline_start
print(f"\n{'='*65}")
print(f"ALL MODELS COMPLETE")
print(f"Total time: {total_elapsed:.0f}s ({total_elapsed/3600:.1f} hr)")
print(f"Results saved under: {OUTPUT_BASE}/{{qwen,gemma,ministral,llama}}/")
print("="*65)


DIPA2 Multimodal — All 4 Models
Dataset: 265 samples | Tasks: T3, T4 | Variants: explicit, no_leak
Models to run: ['qwen', 'gemma', 'ministral', 'llama']

  MODEL: Qwen/Qwen3-VL-8B-Instruct
  Output: /content/drive/MyDrive/DIPA2/multimodal_results/qwen


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

  VRAM: 17.5 GB / 42.4 GB
  Model ready.

  ─ TEMPERATURE: 0.1 ─

  ▶ Task 3 [explicit]
    [   1/265]  11s  avg 10.7s/img
    [  10/265]  67s  avg 6.7s/img
    [  20/265]  124s  avg 6.2s/img
    [  30/265]  189s  avg 6.3s/img
    [  40/265]  247s  avg 6.2s/img
    [  50/265]  315s  avg 6.3s/img
    [  60/265]  371s  avg 6.2s/img
    [  70/265]  439s  avg 6.3s/img
    [  80/265]  495s  avg 6.2s/img
    [  90/265]  555s  avg 6.2s/img
    [ 100/265]  610s  avg 6.1s/img
    [ 110/265]  665s  avg 6.0s/img
    [ 120/265]  720s  avg 6.0s/img
    [ 130/265]  772s  avg 5.9s/img
    [ 140/265]  828s  avg 5.9s/img
    [ 150/265]  885s  avg 5.9s/img
    [ 160/265]  938s  avg 5.9s/img
    [ 170/265]  991s  avg 5.8s/img
    [ 180/265]  1046s  avg 5.8s/img
    [ 190/265]  1100s  avg 5.8s/img
    [ 200/265]  1157s  avg 5.8s/img
    [ 210/265]  1215s  avg 5.8s/img
    [ 220/265]  1273s  avg 5.8s/img
    [ 230/265]  1328s  avg 5.8s/img
    [ 240/265]  1386s  avg 5.8s/img
    [ 250/265]  1446s  avg 5.8s

...

    [   1/265]  3s  avg 3.3s/img
    [  10/265]  32s  avg 3.2s/img
    [  20/265]  63s  avg 3.2s/img
    [  30/265]  95s  avg 3.2s/img
    [  40/265]  126s  avg 3.2s/img
    [  50/265]  157s  avg 3.1s/img
    [  60/265]  188s  avg 3.1s/img
    [  70/265]  219s  avg 3.1s/img
    [  80/265]  250s  avg 3.1s/img
    [  90/265]  282s  avg 3.1s/img
    [ 100/265]  313s  avg 3.1s/img
    [ 110/265]  344s  avg 3.1s/img
    [ 120/265]  375s  avg 3.1s/img
    [ 130/265]  406s  avg 3.1s/img
    [ 140/265]  438s  avg 3.1s/img
    [ 150/265]  470s  avg 3.1s/img
    [ 160/265]  501s  avg 3.1s/img
    [ 170/265]  532s  avg 3.1s/img
    [ 180/265]  564s  avg 3.1s/img
    [ 190/265]  595s  avg 3.1s/img
    [ 200/265]  626s  avg 3.1s/img
    [ 210/265]  658s  avg 3.1s/img
    [ 220/265]  689s  avg 3.1s/img
    [ 230/265]  721s  avg 3.1s/img
    [ 240/265]  752s  avg 3.1s/img
    [ 250/265]  783s  avg 3.1s/img
    [ 260/265]  814s  avg 3.1s/img
    IMAGE — F1: 0.0%  P: 0.0%  R: 0.0%  support=265
    CAPT

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1790: FutureWarning: `hidden_state` is deprecated and will be removed in version v5.20 for `MllamaVisionEncoderLayer.forward`. Use `hidden_states` instead.
  return forward_call(*args, **kwargs)


    [   1/265]  3s  avg 3.2s/img
    [  10/265]  65s  avg 6.5s/img
    [  20/265]  107s  avg 5.4s/img
    [  30/265]  160s  avg 5.3s/img
    [  40/265]  223s  avg 5.6s/img
    [  50/265]  271s  avg 5.4s/img
    [  60/265]  323s  avg 5.4s/img
    [  70/265]  376s  avg 5.4s/img
    [  80/265]  450s  avg 5.6s/img
    [  90/265]  496s  avg 5.5s/img
    [ 100/265]  548s  avg 5.5s/img
    [ 110/265]  595s  avg 5.4s/img
    [ 120/265]  640s  avg 5.3s/img
    [ 130/265]  689s  avg 5.3s/img
    [ 140/265]  735s  avg 5.2s/img
    [ 150/265]  781s  avg 5.2s/img
    [ 160/265]  820s  avg 5.1s/img
    [ 170/265]  874s  avg 5.1s/img
    [ 180/265]  923s  avg 5.1s/img
    [ 190/265]  973s  avg 5.1s/img
    [ 200/265]  1022s  avg 5.1s/img
    [ 210/265]  1069s  avg 5.1s/img
    [ 220/265]  1121s  avg 5.1s/img
    [ 230/265]  1169s  avg 5.1s/img
    [ 240/265]  1214s  avg 5.1s/img
    [ 250/265]  1277s  avg 5.1s/img
    [ 260/265]  1334s  avg 5.1s/img
    Macro F1: 20.62%  P: 35.4%  R: 30.56%
      Dig

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1790: FutureWarning: `hidden_state` is deprecated and will be removed in version v5.20 for `MllamaVisionEncoderLayer.forward`. Use `hidden_states` instead.
  return forward_call(*args, **kwargs)


    [   1/265]  3s  avg 3.4s/img
    [  10/265]  32s  avg 3.2s/img
    [  20/265]  63s  avg 3.2s/img
    [  30/265]  95s  avg 3.2s/img
    [  40/265]  126s  avg 3.1s/img
    [  50/265]  157s  avg 3.1s/img
    [  60/265]  188s  avg 3.1s/img
    [  70/265]  219s  avg 3.1s/img
    [  80/265]  250s  avg 3.1s/img
    [  90/265]  282s  avg 3.1s/img
    [ 100/265]  313s  avg 3.1s/img
    [ 110/265]  344s  avg 3.1s/img
    [ 120/265]  375s  avg 3.1s/img
    [ 130/265]  407s  avg 3.1s/img
    [ 140/265]  438s  avg 3.1s/img
    [ 150/265]  470s  avg 3.1s/img
    [ 160/265]  501s  avg 3.1s/img
    [ 170/265]  532s  avg 3.1s/img
    [ 180/265]  564s  avg 3.1s/img
    [ 190/265]  595s  avg 3.1s/img
    [ 200/265]  626s  avg 3.1s/img
    [ 210/265]  658s  avg 3.1s/img
    [ 220/265]  689s  avg 3.1s/img
    [ 230/265]  720s  avg 3.1s/img
    [ 240/265]  751s  avg 3.1s/img
    [ 250/265]  783s  avg 3.1s/img
    [ 260/265]  814s  avg 3.1s/img
    IMAGE — F1: 0.0%  P: 0.0%  R: 0.0%  support=265
    CAPT